## World Setup

In [ ]:
from dataclasses import asdict
from pprint import pprint
from uniworld import load_pr2_apartment_world
from semantic_digital_twin.world_description.world_entity import Body
import rclpy
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.viz_marker import VizMarkerPublisher
import json
from pycram.datastructures.enums import Arms
from pycram.locations.locations import CostmapLocation
from pycram.motion_executor import simulated_robot
from pycram.plans.factories import sequential
from pycram.robot_plans.actions.core.navigation import NavigateAction
from pycram.robot_plans.actions.core.pick_up import PickUpAction
from pycram.robot_plans.actions.core.robot_body import ParkArmsAction, MoveTorsoAction
from semantic_digital_twin.datastructures.definitions import TorsoState
from dotenv import load_dotenv
from llmr.reasoning.llm_provider import make_llm, LLMProvider
from krrood.entity_query_language.query.match import Match
from llmr.backend import LLMBackend
from pycram.exceptions import MotionDidNotFinish
from llmr.hypotheses import (
    FlanaganFamily,
    FrameNetFamily,
    FrameHypothesisNode,
    FrameRoleHypothesisNode,
    GroundingState,
    HypothesisGraph,
    MotionPhaseHypothesisNode,
    MotionPlanHypothesisNode,
    ProjectionOrchestrator,
    ProjectorRegistry,
)
from llmr.reasoning.flanagan_reasoner import FlanaganReasoner
from llmr.reasoning.framenet_reasoner import FrameNetReasoner
from pycram.datastructures.grasp import GraspDescription
from pycram.robot_plans.actions.core.pick_up import PickUpAction

world, robot_view, context = load_pr2_apartment_world()
context.evaluate_conditions = False

symbol_type = Body

load_dotenv("../.env")

llm = make_llm(LLMProvider.OPENAI, model="gpt-4o-mini", temperature=0.0)
print("LLM ready:", getattr(llm, "model_name", llm))

# HELPERS
def prepare_robot_for_pickup(context):
    setup_plan = sequential(
        [
            ParkArmsAction(Arms.BOTH),
            MoveTorsoAction(TorsoState.HIGH),
        ],
        context=context,
    )
    with simulated_robot:
        setup_plan.perform()


def build_navigate_then_pick(action_instance, context):
    pickup_pose = CostmapLocation(
        target=action_instance.object_designator.global_pose,
        reachable=True,
        reachable_arm=action_instance.arm,
        grasp_description=action_instance.grasp_description,
        context=context,
    ).ground()

    return sequential(
        [
            NavigateAction(pickup_pose, keep_joint_states=True),
            PickUpAction(
                object_designator=action_instance.object_designator,
                arm=action_instance.arm,
                grasp_description=action_instance.grasp_description,
            ),
        ],
        context=context,
    )


def print_pickup_action(action_instance):
    print("Action type:", type(action_instance).__name__)
    print("Object     :", action_instance.object_designator)
    print("Arm        :", action_instance.arm)
    print("Grasp      :", action_instance.grasp_description)
    print("Offset     :", action_instance.grasp_description.manipulation_offset)

def fresh_pickup_match():
    return Match(PickUpAction)(
        object_designator=...,
        arm=...,
        grasp_description=Match(GraspDescription)(
            approach_direction=...,
            vertical_alignment=...,
            manipulator=...,
        ),
    )

# Graph Setup
graph = HypothesisGraph()
registry = ProjectorRegistry([
    FrameNetFamily.make_projector(),
    FlanaganFamily.make_projector(),
])
manager = ProjectionOrchestrator(graph=graph, registry=registry)

def run_instruction(instruction: str):
    backend = LLMBackend(
        llm=llm,
        symbol_type=symbol_type,
        instruction=instruction,
        strict_required=True,
        reasoners=[
            FrameNetReasoner(llm=llm),
            FlanaganReasoner(llm=llm),
        ],
        hypothesis_graph_manager=manager,
    )
    action = next(iter(backend.evaluate(fresh_pickup_match())))
    return action, backend

def role_rows(roles):
    return [
        {
            "display_id": role.display_id,
            "role": role.role_name,
            "family": role.role_family,
            "text": role.filler_text,
            "kind": role.filler_kind,
            "status": role.meta.status.value,
            "grounding": role.meta.grounding.value,
            "run_id": role.meta.short_run_id,
        }
        for role in roles
    ]


def phase_rows(phases):
    return [
        {
            "display_id": phase.display_id,
            "index": phase.phase_index,
            "phase": phase.phase_name,
            "target": phase.target_object,
            "contact": phase.contact,
            "motion_type": phase.motion_type,
            "urgency": phase.urgency,
            "run_id": phase.meta.short_run_id,
        }
        for phase in phases
    ]

# ROS setup
rclpy.init()
_ros_node = rclpy.create_node('semantic_digital_twin')
import threading
_ros_thread = threading.Thread(target=rclpy.spin, args=(_ros_node,), daemon=True)
_ros_thread.start()


framenet = FrameNetFamily.make_view(graph)
flanagan = FlanaganFamily.make_view(graph)

In [ ]:
_tf_publisher = TFPublisher(_world=world, node=_ros_node)
_viz_publisher = VizMarkerPublisher(_world=world, node=_ros_node)
print("ROS2 publishers started")

## Inferences

In [ ]:
INSTRUCTION = "pick up the milk from the table"

In [ ]:
action, backend = run_instruction(INSTRUCTION)

In [ ]:
print(json.dumps(backend.semantics.model_dump(), indent=2))

## Execution

In [ ]:
prepare_robot_for_pickup(context)

role1_plan_node = build_navigate_then_pick(action, context)
print("Role 1 Navigate + PickUp plan ready:", role1_plan_node)
with simulated_robot:
    try:
        role1_plan_node.perform()
    except MotionDidNotFinish:
        print("Role 1 motion did not finish; pickup parameters and plan construction were still exercised.")


## Query

In [ ]:
from krrood.entity_query_language.factories import an, entity, variable
action_graph = graph.subgraph_for_action(action)
action_framenet = FrameNetFamily.make_view(action_graph)
action_flanagan = FlanaganFamily.make_view(action_graph)
frame = variable(FrameHypothesisNode, domain=action_framenet.frames())
role = variable(FrameRoleHypothesisNode, domain=action_framenet.roles())
plan = variable(MotionPlanHypothesisNode, domain=action_flanagan.plans())
phase = variable(MotionPhaseHypothesisNode, domain=action_flanagan.phases())


In [ ]:
# Query 1: inferred FrameNet frame for this instruction
frame_query = an(
    entity(frame).where(
        frame.instruction_text == INSTRUCTION,
    )
)
inferred_frames = list(frame_query.evaluate())
pprint(asdict(inferred_frames[0]))

In [ ]:
# Query 2: grounded FrameNet roles
grounded_role_query = an(
    entity(role).where(
        role.meta.grounding == GroundingState.SYMBOL_GROUNDED,
    )
)
role_rows(list(grounded_role_query.evaluate()))


In [ ]:
# list(grounded_role_query.evaluate())

In [ ]:
# Query 3: all Flanagan motion phases
phase_query = an(entity(phase))
phase_rows(list(phase_query.evaluate()))


In [ ]:
# Query 4: contact phases only
contact_phase_query = an(
    entity(phase).where(
        phase.contact == True,
    )
)
phase_rows(list(contact_phase_query.evaluate()))


In [ ]:
# Query 5: compare FrameNet theme fillers with Flanagan phase targets
theme_query = an(
    entity(role).where(
        role.role_name == "theme",
    )
)
theme_roles = list(theme_query.evaluate())
phase_targets = sorted({item.target_object for item in action_flanagan.phases()})

{
    "framenet_theme_texts": [item.filler_text for item in theme_roles],
    "flanagan_phase_targets": phase_targets,
    "shared_targets": sorted(set(item.filler_text for item in theme_roles) & set(phase_targets)),
}


## HypothesisGraphs

In [ ]:
from graphviz import Source
from IPython.display import display

try:
    display(Source(graph.to_dot(), format="svg"))
except Exception:
    print(graph.to_dot())


In [ ]:
# action_graph = graph.subgraph_for_reasoner("framenet_reasoner")
# from graphviz import Source
# from IPython.display import display
#
# try:
#     display(Source(action_graph.to_dot(), format="svg"))
# except Exception:
#     print(action_graph.to_dot())


In [ ]:
fv = FrameNetFamily.make_view(graph).action_subgraph(action)

In [ ]:
fv.roles()

In [ ]:
fv.grounded_roles()[0]

In [ ]:
fv.supported_roles()

In [ ]:
fv.hypothesis_only_roles()

In [ ]:
from llmr.hypotheses.elements import HypothesisNode
frame = variable(HypothesisNode, domain=fv.frames())

In [ ]:
frame_results = list(an(entity(frame)).evaluate())

In [ ]:
asdict(frame_results[0])